In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
import xml.etree.ElementTree as ET

tree = ET.parse("./v1.4/dev/archehr-qa.xml")
root = tree.getroot()

In [7]:
from pprint import pprint

data = []

for case in root.findall("case"):
    case_id = case.get("id")
    clinician_question = case.findtext("clinician_question", default="").strip()
    patient_question = case.findtext("patient_narrative", default="").strip()
    notes = case.findtext("note_excerpt", default="").strip()
    item = {
        "case_id": case_id,
        "clinician_question": clinician_question,
        "patient_question": patient_question,
        "notes": notes,
    }
    data.append(item)

pprint(data[0])

{'case_id': '1',
 'clinician_question': 'Why was ERCP recommended to him over continuing a '
                       'medication-based treatment?',
 'notes': 'Brief Hospital Course:\n'
          '\n'
          'During the ERCP a pancreatic stent was required to facilitate\n'
          'access to the biliary system (removed at the end of the\n'
          'procedure), and a common bile duct stent was placed to allow\n'
          'drainage of the biliary obstruction caused by stones and sludge.\n'
          "However, due to the patient's elevated INR, no sphincterotomy or\n"
          'stone removal was performed. Frank pus was noted to be draining\n'
          'from the common bile duct, and post-ERCP it was recommended that\n'
          'the patient remain on IV Zosyn for at least a week. The\n'
          'Vancomycin was discontinued.\n'
          '\n'
          'On hospital day 4 (post-procedure day 3) the patient returned to\n'
          'ERCP for re-evaluation of her biliary stent as 

In [9]:
from prompts import EXTRACTIVE_PROMPT, CONTROLLED_SYNTHESIS_PROMPT

def extract(question, notes):
    messages = [
        {"role": "system", "content": EXTRACTIVE_PROMPT},
        {"role": "user", "content": f"Clinician-Interpreted Question:\n{question}\n\nClinical Note Excerpt:\n{notes}\n\nRelevant Sentences:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

def synthesize(question, sentences):
    messages = [
        {"role": "system", "content": CONTROLLED_SYNTHESIS_PROMPT},
        {"role": "user", "content": f"Patient Question:\n{question}\n\nRelevant Sentences:\n{sentences}\n\nAnswer:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [10]:
from tqdm.notebook import tqdm

answers = []

for i in tqdm(data):
    s = extract(i['clinician_question'], i['notes'])
    answer = synthesize(i['patient_question'], s)
    answer_dict = {
        'case_id': i['case_id'],
        'answer': answer
    }
    answers.append(answer_dict)

  0%|          | 0/20 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [12]:
submission = [{'case_id': i['case_id'], 'prediction': i['answer']} for i in answers]

In [15]:
import json

with open("./submission.json", 'w') as f:
    json.dump(submission, f)